# Coding Attention Mechanism

### Simple attention mechanism

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch

In [3]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], ## your
        [0.55, 0.87, 0.66], # journey
        [0.57, 0.85, 0.64], # starts
        [0.22, 0.58, 0.33], # with
        [0.77, 0.25, 0.10], # one
        [0.05, 0.80, 0.55] # steps
    ]
)

In [4]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [5]:
input_1 = inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [6]:
torch.dot(input_query, input_1)

tensor(0.9544)

In [7]:
for i in range(len(inputs)):
    res = torch.dot(inputs[i], input_query)
    print(res)

tensor(0.9544)
tensor(1.4950)
tensor(1.4754)
tensor(0.8434)
tensor(0.7070)
tensor(1.0865)


In [8]:
query = inputs[1]

attn_score = torch.empty(inputs.shape[0])

for i, i_x in enumerate(inputs):
    attn_score[i] = torch.dot(i_x, query)

attn_score = torch.softmax(attn_score, dim=0)
attn_score

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [9]:
query = inputs[1]

context_vec = torch.zeros(query.shape)

for i, i_x in enumerate(inputs):
    context_vec += attn_score[i] * i_x
print(context_vec)

tensor([0.4419, 0.6515, 0.5683])


### Simple attention without trainable parameter

In [33]:
attn_scores = inputs @ inputs.T
attn_weight = torch.softmax(attn_scores, dim=1)
context_vec = attn_weight @ inputs
print(context_vec)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### With trainable weight

In [34]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [68]:
x_2 = inputs[1]
in_d = inputs.shape[1]
out_d = 2

In [69]:
torch.manual_seed(123)

w_query = torch.nn.Parameter(torch.rand(in_d, out_d))
w_key = torch.nn.Parameter(torch.rand(in_d, out_d))
w_value = torch.nn.Parameter(torch.rand(in_d, out_d))

In [70]:
query = x_2 @ w_query
query

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [71]:
key = inputs @ w_key
value = inputs @ w_value

In [72]:
key.shape

torch.Size([6, 2])

In [73]:
attn_score = torch.dot(query, key[1])
attn_score

tensor(1.8524, grad_fn=<DotBackward0>)

In [74]:
attn_scores = torch.matmul(query, key.T)
attn_scores

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [75]:
d_k = key.shape[1]


attn_weight_2 = torch.softmax(attn_scores / d_k**0.5, dim=-1)
attn_weight_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [76]:
context_vec_2 = attn_weight_2 @ value
context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

## Implementing a compactable class

In [79]:
class SelfAttentionV1(torch.nn.Module):

    def __init__(self, in_d, out_d):
        super().__init__()
        self.w_query = torch.nn.Parameter(torch.rand(in_d, out_d))
        self.w_key = torch.nn.Parameter(torch.rand(in_d, out_d))
        self.w_value = torch.nn.Parameter(torch.rand(in_d, out_d))


    def forward(self):
        query = inputs @ self.w_query
        key = inputs @ self.w_key
        value = inputs @ self.w_value

        attn_scores = torch.matmul(query, key.T)
        attn_weight = torch.softmax(attn_scores / key.shape[1]**0.5, dim=-1)
        context_vec = attn_weight @ value
        return context_vec

torch.manual_seed(123)
attention = SelfAttentionV1(in_d, out_d)
attention()

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [87]:
class SelfAttentionV2(torch.nn.Module):

    def __init__(self, in_d, out_d, qkv_bias = False):
        super().__init__()
        self.w_query = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.w_key = torch.nn.Linear(in_d, out_d, bias=qkv_bias)
        self.w_value = torch.nn.Linear(in_d, out_d, bias=qkv_bias)


    def forward(self, inputs):
        query = self.w_query(inputs)
        key = self.w_key(inputs)
        value = self.w_value(inputs)

        attn_scores = torch.matmul(query, key.T)
        attn_weight = torch.softmax(attn_scores / key.shape[1]**0.5, dim=-1)
        context_vec = attn_weight @ value
        return context_vec

torch.manual_seed(789)
attention = SelfAttentionV2(in_d, out_d)
attention(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)